In [45]:
import http.client
import time
import urllib.error

import numpy as np
import matplotlib.pyplot as plt

# provides access to FITS files
# a standard format for storing astronomical data, including images and tables
# source: https://docs.astropy.org/en/stable/io/fits/index.html
from astropy.io import fits

# describes transformations between different coordinate sets
# can be used to convert from pixel coordinates to world coordinates and vice versa
# source: https://docs.astropy.org/en/latest/wcs/index.html
from astropy.wcs import WCS

# tools for working with tabular data using an interface similar to numpy
# i.e. structured arrays
# source: https://docs.astropy.org/en/stable/table/index.html
from astropy.table import Table

# helps us account for physical quantities
# source: https://docs.astropy.org/en/stable/units/index.html
from astropy import units as u

# class that provides a way to represent and manipulate celestial coordinates
# and convert between coordinate systems
from astropy.coordinates import SkyCoord

# IRSA queries is a module which "provides access to the public astrophysics catalogs, 
# images, and spectra curated by the NASA/IPAC Infrared Science Archive (IRSA) at Caltech"
# source: https://docs.astropy.org/projects/astroquery/en/latest/ipac/irsa/irsa.html
from astroquery.ipac.irsa import Irsa

# web framework for astronomical data access and visualization, developed at Caltech
# source: https://caltech-ipac.github.io/firefly_client/
from firefly_client import FireflyClient

# The time it takes to read SPHEREx files can exceed
# astropy's default timeout limit. Increase it.
from astropy.utils.data import conf
conf.remote_timeout = 120

In [46]:
# from tutorial, just testing

ra_deg = 304.693508808
dec_deg = 42.4436872991

coord = SkyCoord(ra_deg, dec_deg, unit='deg')
search_radius = 1 * u.arcsec

In [51]:
# returns info of available IRSA SIA collections, filtered for all the ones
# containing "spherex" in their name
Irsa.list_collections(servicetype='SIA', filter='spherex')


collection
object
spherex_dr1
spherex_dr1_cube
spherex_dr1_deep
spherex_qr3
spherex_qr3_deep


In [56]:
# Use standard SIA2 attributes to query the IRSA SIA service.

results = Irsa.query_sia(pos=(coord, search_radius), collection=['spherex_dr1', 'spherex_qr3'], t_min='2023-01-01', t_max='2023-12-31')

spectral_image_url = results['access_url'][0]
right_as = results['s_ra'][0]
deccy = results['s_dec'][0]
time_min = results['t_min'][0]
time_max = results['t_max'][0]

print(time_min, time_max)

print(spectral_image_url)
print(right_as)
print(deccy)

DALQueryError: UsageFault: BAD_REQUEST: Unknown parameter: t_max

In [53]:
# Max number of times to retry HTTP errors.

hdulist = fits.open(spectral_image_url, memmap=False)

max_retries = 3
for attempt in range(max_retries):
    try:
        # Load the data, reading every extension into memory so that
        # hdulist stays usable after the file is closed.
        for hdu in hdulist:
            hdu.data
        break
    except (TimeoutError, urllib.error.HTTPError, http.client.IncompleteRead):
        if attempt == max_retries - 1:
            raise
        time.sleep(10 * (attempt + 1))


hdulist.info()

Filename: C:\Users\accos\.cache\astropy\download\url\2edb0eda84f44a114b0a0118c29fe067\contents
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  IMAGE         1 ImageHDU       293   (2040, 2040)   float32   
  2  FLAGS         1 CompImageHDU    185   (2040, 2040)   int32   
  3  VARIANCE      1 ImageHDU       119   (2040, 2040)   float32   
  4  ZODI          1 ImageHDU       122   (2040, 2040)   float32   
  5  EPSF          1 BinTableHDU     56   441R x 11C   ['J', 'J', 'E', 'E', 'E', 'E', 'J', 'E', 'E', 'E', '1089D']   
  6  WCS-WAVE      1 BinTableHDU     22   1R x 3C   ['9J', '9J', '162E']   


In [54]:
# opens web browser to this link

fc = FireflyClient.make_client(url="https://irsa.ipac.caltech.edu/irsaviewer")

fc.show_fits_image(file_input=spectral_image_url,
             plot_id="spectral_image",
             Title="Spectral Image"
             )

fc.reinit_viewer()

{'success': True}